# 🌟 Kaju - AI Voice Assistant (Dual Mode: Voice/Text In ➡️ Voice Out)
This notebook implements the complete **Kaju Voice Assistant** supporting both **Text Input** and **Microphone Voice Input (Enter to Start ➡️ Enter to Stop)**, with natural **Female Voice Output**.

### 🚀 Architecture Highlights:
- **Voice Input (STT)**: Groq Whisper (`whisper-large-v3-turbo`) with manual Start/Stop recording.
- **Brain (LLM)**: Groq API (`openai/gpt-oss-120b`).
- **Voice Output (TTS)**: Microsoft Edge Neural TTS (`edge-tts`) - **100% Free & Unlimited**.
- **Offline Backup**: Windows SAPI5 (`pyttsx3`) female voice.

In [1]:
# 1. Install required packages
!pip install requests edge-tts pygame pyttsx3 sounddevice scipy numpy nest_asyncio

In [17]:
# 3. Configuration & Female Voice Selection

# Female Voice Options:
# - "en-US-AvaNeural"     : Natural, expressive US English female (Default)
# - "en-US-EmmaNeural"    : Warm & friendly US English female
# - "en-IN-NeerjaNeural"  : Indian English female voice
# - "en-GB-SoniaNeural"   : British English female voice
FEMALE_VOICE = "en-US-AvaNeural"

# Load API Key from 'Groq api key.txt' if present, or specify directly
if os.path.exists("Groq api key.txt"):
    with open("Groq api key.txt", "r", encoding="utf-8") as f:
        API_KEY = f.read().strip()
else:
    API_KEY = ""

LLM_MODEL = "openai/gpt-oss-120b"
STT_MODEL = "whisper-large-v3-turbo"
CHAT_URL = "https://api.groq.com/openai/v1/chat/completions"
STT_URL = "https://api.groq.com/openai/v1/audio/transcriptions"

print(f"Selected Female Voice : {FEMALE_VOICE}")
print(f"LLM Model             : {LLM_MODEL}")
print(f"STT Model             : {STT_MODEL}")

Selected Female Voice : en-US-AvaNeural
LLM Model             : openai/gpt-oss-120b
STT Model             : whisper-large-v3-turbo


In [18]:
# 4. System Instruction for Kaju
SYSTEM_INSTRUCTION = """
You are Kaju, a friendly, intelligent, and helpful female voice assistant.
Guidelines:
- Always respond in a warm, polite, and natural conversational tone.
- Keep your answers concise, clear, and easy to understand when spoken aloud (usually 1-3 sentences unless asked for more).
- Avoid bullet lists or markdown symbols unless necessary, because your response is converted directly into speech.
- If asked who you are, introduce yourself as Kaju.
"""

chat_history = [{"role": "system", "content": SYSTEM_INSTRUCTION}]
print("System instruction initialized.")

System instruction initialized.


In [19]:
# 5. Speech-To-Text (Voice Input) Module (Enter to Start & Enter to Stop)

def record_microphone() -> str:
    """Records audio from microphone until user presses [ENTER] to stop."""
    temp_wav = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
    temp_wav_path = temp_wav.name
    temp_wav.close()

    try:
        device_info = sd.query_devices(kind='input')
        channels = min(2, max(1, device_info.get('max_input_channels', 1)))
        sample_rate = int(device_info.get('default_samplerate', 44100))

        audio_queue = queue.Queue()

        def audio_callback(indata, frames, time_info, status):
            audio_queue.put(indata.copy())

        print("\n" + "-" * 55)
        print(" 🎙️  RECORDING STARTED! Speak into your microphone...")
        print(" 👉 Press [ENTER] when you are finished speaking to STOP")
        print("-" * 55)

        stream = sd.InputStream(samplerate=sample_rate, channels=channels, dtype='int16', callback=audio_callback)
        with stream:
            input()  # Wait for user to press ENTER to stop recording

        print("⚡ Recording STOPPED. Transcribing your voice...")

        chunks = []
        while not audio_queue.empty():
            chunks.append(audio_queue.get())

        if not chunks:
            print("[Warning] No audio was captured.")
            if os.path.exists(temp_wav_path):
                os.remove(temp_wav_path)
            return ""

        audio_data = np.concatenate(chunks, axis=0)
        wav.write(temp_wav_path, sample_rate, audio_data)
        return temp_wav_path

    except Exception as e:
        print(f"[Microphone Error] {e}")
        if os.path.exists(temp_wav_path):
            os.remove(temp_wav_path)
        return ""

def transcribe_audio(audio_path: str, api_key: str = API_KEY) -> str:
    """Transcribes audio using Groq's high-speed Whisper AI model."""
    if not audio_path or not os.path.exists(audio_path):
        return ""

    headers = {"Authorization": f"Bearer {api_key}"}
    try:
        with open(audio_path, "rb") as f:
            files = {"file": (os.path.basename(audio_path), f, "audio/wav")}
            data = {"model": STT_MODEL}
            response = requests.post(STT_URL, headers=headers, files=files, data=data, timeout=15)
            result = response.json()
            return result.get("text", "").strip()
    except Exception as e:
        print(f"[STT Connection Error] {e}")
        return ""
    finally:
        if os.path.exists(audio_path):
            try:
                os.remove(audio_path)
            except Exception:
                pass

print("Voice Input (STT) Module ready!")

Voice Input (STT) Module ready!


In [20]:
# 6. Text-To-Speech (Voice Output) Module

def run_async(coroutine):
    """Runs async coroutines cleanly inside Jupyter notebooks."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coroutine)

def clean_text_for_speech(text: str) -> str:
    """Cleans markdown symbols and emojis for smooth pronunciation."""
    cleaned = re.sub(r'[*_#`~]', '', text)
    cleaned = re.sub(r'[^\x00-\x7F]+', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

async def _synthesize_edge_tts(text: str, voice: str, output_path: str):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)

def speak_offline_fallback(text: str):
    """Fallback TTS using Windows built-in SAPI5 female voice."""
    try:
        engine = pyttsx3.init()
        voices = engine.getProperty('voices')
        for v in voices:
            if "female" in v.name.lower() or "zira" in v.name.lower() or "heera" in v.name.lower():
                engine.setProperty('voice', v.id)
                break
        engine.setProperty('rate', 175)
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        print(f"[Offline TTS Warning] {e}")

def speak(text: str, voice: str = FEMALE_VOICE):
    """Speaks text using 100% Free Edge Neural Female Voice."""
    clean_text = clean_text_for_speech(text)
    if not clean_text:
        return

    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    temp_path = temp_file.name
    temp_file.close()

    try:
        run_async(_synthesize_edge_tts(clean_text, voice, temp_path))

        if not pygame.mixer.get_init():
            pygame.mixer.init()
        
        pygame.mixer.music.load(temp_path)
        pygame.mixer.music.play()

        while pygame.mixer.music.get_busy():
            time.sleep(0.05)

        pygame.mixer.music.stop()
        pygame.mixer.music.unload()

    except Exception:
        speak_offline_fallback(clean_text)

    finally:
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except Exception:
                pass

print("Voice Output (TTS) Engine ready!")

Voice Output (TTS) Engine ready!


In [23]:
# 7. Function to Ask Kaju (Query ➡️ LLM Response ➡️ Print & Speak)

def ask_kaju(message: str, voice: str = FEMALE_VOICE):
    """Sends text to Kaju, prints reply, and speaks in natural female voice."""
    if not message.strip():
        return None

    chat_history.append({"role": "user", "content": message})
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": LLM_MODEL,
        "messages": chat_history,
        "temperature": 0.7,
        "max_tokens": 300
    }
    
    try:
        response = requests.post(CHAT_URL, headers=headers, json=payload, timeout=20)
        data = response.json()

        if "error" in data:
            print("API Error:", data["error"])
            return None

        reply = data["choices"][0]["message"]["content"].strip()
        chat_history.append({"role": "assistant", "content": reply})
        
        # 1. Print Text Response
        print(f"\nKaju: {reply}")
        
        # 2. Speak Response Aloud
        speak(reply, voice=voice)
        return reply

    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
# 8. Interactive Dual-Mode Chat (Type text OR press Enter for Microphone Voice Input)
print("=" * 65)
print(" 🎙️ KAJU DUAL-MODE VOICE ASSISTANT")
print(" - Type your text message OR press Enter for Voice Input 🎙️")
print(" - In Voice Mode: Speak as long as you want, then press [ENTER] to stop")
print(" - Type 'exit', 'quit', or 'bye' to stop.")
print("=" * 65)

# Greeting
greeting = "Hello! I am Kaju. You can type or speak to me anytime!"
print(f"\nKaju: {greeting}")
speak(greeting)

while True:
    try:
        prompt_str = input("\nYou [Type text OR press Enter for Voice 🎙️]: ").strip()

        if prompt_str.lower() in ["quit", "exit", "bye", "stop"]:
            farewell = "Goodbye! Have a wonderful day! 👋"
            print(f"\nKaju: {farewell}")
            speak(farewell)
            break

        # If empty (just hit Enter) or typed 'v' -> Record from Microphone
        if prompt_str == "" or prompt_str.lower() in ["v", "voice", "mic", "speak"]:
            audio_file = record_microphone()
            if not audio_file:
                continue
            
            user_text = transcribe_audio(audio_file)
            if not user_text:
                print("Could not detect clear speech. Please try again.")
                continue
                
            print(f"\n🗣️  You (Voice): \"{user_text}\"")
            ask_kaju(user_text)
        else:
            # User typed regular text
            ask_kaju(prompt_str)

    except (KeyboardInterrupt, EOFError):
        print("\nConversation ended.")
        break

 🎙️ KAJU DUAL-MODE VOICE ASSISTANT
 - Type your text message OR press Enter for Voice Input 🎙️
 - In Voice Mode: Speak as long as you want, then press [ENTER] to stop
 - Type 'exit', 'quit', or 'bye' to stop.

Kaju: Hello! I am Marghi. You can type or speak to me anytime!


In [ ]:
# 9. Save and Inspect Chat History
with open("kaju_chat_history.json", "w", encoding="utf-8") as f:
    json.dump(chat_history, f, indent=2)

print("Chat history saved to 'kaju_chat_history.json'!")